In [7]:
import zipfile
import os

zip_file_path = '/content/Internship dataset 1.zip'
extraction_path = '/content/'

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_path)

print(f"'{zip_file_path}' extracted to '{extraction_path}'")
print("Files in extraction path:")
print(os.listdir(extraction_path))
df = pd.read_csv(f"{extraction_path}Bengaluru_House_Data.csv")
print("Dataset loaded successfully")
df = pd.read_csv(f"{extraction_path}Bengaluru_House_Data.csv")
print("Dataset loaded successfully")
display(df.head())

'/content/Internship dataset 1.zip' extracted to '/content/'
Files in extraction path:
['.config', 'Bengaluru_House_Data.csv', 'Internship dataset 1.zip', 'sample_data']
Dataset loaded successfully
Dataset loaded successfully


,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [8]:
#Check data types and non - null counts
df.info()

#Check missing values in each column
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     13320 non-null  object 
 1   availability  13320 non-null  object 
 2   location      13319 non-null  object 
 3   size          13304 non-null  object 
 4   society       7818 non-null   object 
 5   total_sqft    13320 non-null  object 
 6   bath          13247 non-null  float64
 7   balcony       12711 non-null  float64
 8   price         13320 non-null  float64
dtypes: float64(3), object(6)
memory usage: 936.7+ KB
area_type          0
availability       0
location           1
size              16
society         5502
total_sqft         0
bath              73
balcony          609
price              0
dtype: int64


In [9]:
# Keep only essential columns for prediction
df2 = df[['location', 'size', 'total_sqft', 'bath', 'price']].copy()

# Drop rows where critical fields (like size or bath) are missing
df3 = df2.dropna()
print("Shape after dropping NaNs:", df3.shape)

Shape after dropping NaNs: (13246, 5)


In [10]:
# Extract the first number from the 'size' string
df3['bhk'] = df3['size'].apply(lambda x: int(x.split(' ')[0]))
df3.head()

/tmp/ipykernel_4486/1677806114.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df3['bhk'] = df3['size'].apply(lambda x: int(x.split(' ')[0]))


,location,size,total_sqft,bath,price,bhk
0,Electronic City Phase II,2 BHK,1056,2.0,39.07,2
1,Chikka Tirupathi,4 Bedroom,2600,5.0,120.00,4
2,Uttarahalli,3 BHK,1440,2.0,62.00,3
3,Lingadheeranahalli,3 BHK,1521,3.0,95.00,3
4,Kothanur,2 BHK,1200,2.0,51.00,2


In [13]:
def convert_sqft_to_num(x):
    tokens = str(x).split('-')
    if len(tokens) == 2:
        return (float(tokens[0]) + float(tokens[1])) / 2
    try:
        return float(x)
    except:
        return None

df4 = df3.copy()
df4['total_sqft'] = df4['total_sqft'].apply(convert_sqft_to_num)

# Drop any rows where total_sqft couldn't be converted
df4 = df4.dropna()

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Define features (X) and target variable (y)
X = df4[['total_sqft', 'bath', 'bhk']]
y = df4['price']

# Split into 80% training data and 20% testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Evaluate model accuracy (R^2 Score)
score = model.score(X_test, y_test)
print(f"Model Accuracy (R² Score): {score:.4f}")

Model Accuracy (R² Score): 0.4498


In [15]:
# Predict price for [total_sqft, bath, bhk]
sample_house = [[1000, 2, 2]]
predicted_price = model.predict(sample_house)
print(f"Predicted Price: ₹{predicted_price[0]:.2f} Lakhs")

Predicted Price: ₹60.33 Lakhs


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
